# PowerGraphBuilder 预处理/后处理结果展示（test.ipynb）

本 notebook 展示：
- 原始 OPFData JSON 的结构与关键字段（前处理前）
- `power_graph_builder.py` 生成的图数据（后处理结果）
- （可选）标准化前后特征分布变化

In [1]:
from pathlib import Path
import json
import numpy as np

from power_graph_builder import (
    PowerGraphBuilder,
    build_graph_from_json,
    NODE_RAW_MAX, NODE_TYPE_DIM, NODE_FEAT_DIM,
    EDGE_RAW_MAX, EDGE_TYPE_DIM, EDGE_FEAT_DIM,
)

OPFDATA_DIR = Path("power_demo_work/opfdata")
SAMPLE_JSON = next(OPFDATA_DIR.rglob("*.json"), None)

print("OPFDATA_DIR:", OPFDATA_DIR.resolve())
print("SAMPLE_JSON :", SAMPLE_JSON)

if SAMPLE_JSON is None:
    raise FileNotFoundError(
        "未找到 power_demo_work/opfdata 下的 *.json。\n"
        "请先在 repo 根目录运行：python download_dataset.py"
    )

OPFDATA_DIR: /Users/jumiray/Projects/power-graph-risk-learning/power_demo_work/opfdata
SAMPLE_JSON : power_demo_work/opfdata/dataset_release_1__pglib_opf_case14_ieee_0_extracted/gridopt-dataset-tmp/dataset_release_1/pglib_opf_case14_ieee/group_0/example_4.json


In [2]:
with open(SAMPLE_JSON, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Top-level keys:", list(sample.keys()))
print("grid keys      :", list(sample.get("grid", {}).keys()))
print("solution keys  :", list(sample.get("solution", {}).keys()))
print("metadata keys  :", list(sample.get("metadata", {}).keys()))

grid_nodes = sample["grid"]["nodes"]
grid_edges = sample["grid"]["edges"]

print("\n[grid.nodes] element types:", list(grid_nodes.keys()))
for k, v in grid_nodes.items():
    print(f"  - {k:10s} count={len(v):5d}  feat_dim={(len(v[0]) if len(v)>0 else 0)}")

print("\n[grid.edges] edge types:", list(grid_edges.keys()))
def edge_count(ed):
    s = ed.get("senders", [])
    r = ed.get("receivers", [])
    return max(len(s), len(r))

for k, ed in grid_edges.items():
    if isinstance(ed, dict):
        cnt = edge_count(ed)
        feat_dim = len(ed.get("features", [[ ]])[0]) if len(ed.get("features", []))>0 else 0
        print(f"  - {k:15s} count={cnt:5d} feat_dim={feat_dim}")
    else:
        print(f"  - {k:15s} (unexpected format)")

print("\nLabel (metadata.objective) =", sample.get("metadata", {}).get("objective"))

Top-level keys: ['grid', 'solution', 'metadata']
grid keys      : ['nodes', 'edges', 'context']
solution keys  : ['nodes', 'edges']
metadata keys  : ['objective']

[grid.nodes] element types: ['bus', 'generator', 'load', 'shunt']
  - bus        count=   14  feat_dim=4
  - generator  count=    5  feat_dim=11
  - load       count=   11  feat_dim=2
  - shunt      count=    1  feat_dim=2

[grid.edges] edge types: ['ac_line', 'transformer', 'generator_link', 'load_link', 'shunt_link']
  - ac_line         count=   17 feat_dim=9
  - transformer     count=    3 feat_dim=11
  - generator_link  count=    5 feat_dim=0
  - load_link       count=   11 feat_dim=0
  - shunt_link      count=    1 feat_dim=0

Label (metadata.objective) = 1973.1230176731374


In [3]:
g = build_graph_from_json(
    SAMPLE_JSON,
    normalize_features=False,
    include_solution=True,
    include_links=True,
)

is_dict = isinstance(g, dict)
print("Graph type:", "dict" if is_dict else type(g))

if is_dict:
    x = np.array(g["x"], dtype=float)
    edge_index = np.array(g["edge_index"], dtype=int)
    edge_attr = np.array(g["edge_attr"], dtype=float)
    y = g["y"]
    sol_node = np.array(g["sol_node"], dtype=float)
    sol_edge = np.array(g["sol_edge"], dtype=float)
    meta = g["meta"]
else:
    # torch_geometric.data.Data
    x = g.x.detach().cpu().numpy()
    edge_index = g.edge_index.detach().cpu().numpy()
    edge_attr = g.edge_attr.detach().cpu().numpy()
    y = float(g.y.item())
    sol_node = g.sol_node.detach().cpu().numpy()
    sol_edge = g.sol_edge.detach().cpu().numpy()
    meta = g.meta

print("\n[Graph summary]")
print("meta:", meta)
print("x.shape        :", x.shape, " expected:", (meta["n_nodes"], NODE_FEAT_DIM))
print("edge_index.shape:", edge_index.shape, " (should be 2 x E)")
print("edge_attr.shape :", edge_attr.shape, " expected:", (meta["n_edges"], EDGE_FEAT_DIM))
print("sol_node.shape  :", sol_node.shape)
print("sol_edge.shape  :", sol_edge.shape)
print("y (objective)   :", y)

Graph type: dict

[Graph summary]
meta: {'source_file': 'power_demo_work/opfdata/dataset_release_1__pglib_opf_case14_ieee_0_extracted/gridopt-dataset-tmp/dataset_release_1/pglib_opf_case14_ieee/group_0/example_4.json', 'n_bus': 14, 'n_gen': 5, 'n_load': 11, 'n_shunt': 1, 'n_nodes': 31, 'n_edges': 37}
x.shape        : (31, 15)  expected: (31, 15)
edge_index.shape: (2, 37)  (should be 2 x E)
edge_attr.shape : (37, 14)  expected: (37, 14)
sol_node.shape  : (31, 2)
sol_edge.shape  : (37, 4)
y (objective)   : 1973.1230176731374


In [4]:
# x = [raw_padded(11) | node_type_onehot(4)]
raw_x = x[:, :NODE_RAW_MAX]
type_x = x[:, NODE_RAW_MAX:NODE_RAW_MAX+NODE_TYPE_DIM]

node_type_names = ["bus", "generator", "load", "shunt"]
node_type_id = type_x.argmax(axis=1)
node_type_count = {node_type_names[i]: int((node_type_id==i).sum()) for i in range(len(node_type_names))}

print("Node type counts:", node_type_count)

print("\nFirst 5 nodes (raw part + type):")
for i in range(min(5, x.shape[0])):
    print(i, raw_x[i, :6], " type=", node_type_names[node_type_id[i]])

Node type counts: {'bus': 14, 'generator': 5, 'load': 11, 'shunt': 1}

First 5 nodes (raw part + type):
0 [1.   3.   0.94 1.06 0.   0.  ]  type= bus
1 [1.   2.   0.94 1.06 0.   0.  ]  type= bus
2 [1.   2.   0.94 1.06 0.   0.  ]  type= bus
3 [1.   1.   0.94 1.06 0.   0.  ]  type= bus
4 [1.   1.   0.94 1.06 0.   0.  ]  type= bus


In [5]:
# edge_attr = [raw_padded(11) | edge_type_onehot(3)]
raw_e = edge_attr[:, :EDGE_RAW_MAX]
type_e = edge_attr[:, EDGE_RAW_MAX:EDGE_RAW_MAX+EDGE_TYPE_DIM]

edge_type_names = ["ac_line", "transformer", "link"]
edge_type_id = type_e.argmax(axis=1)
edge_type_count = {edge_type_names[i]: int((edge_type_id==i).sum()) for i in range(len(edge_type_names))}

print("Edge type counts:", edge_type_count)

# link 边 raw 特征应当全是 0（因为 LINK_FEAT_DIM=0，pad 出来全0）
link_mask = (edge_type_id == 2)
if link_mask.any():
    print("\nLink edges raw feature abs-sum (should be 0):",
          float(np.abs(raw_e[link_mask]).sum()))
else:
    print("\nNo link edges (include_links=False or JSON has none).")

Edge type counts: {'ac_line': 17, 'transformer': 3, 'link': 17}

Link edges raw feature abs-sum (should be 0): 0.0


In [6]:
grid_nodes = sample["grid"]["nodes"]
n_bus = len(grid_nodes.get("bus", []))
n_gen = len(grid_nodes.get("generator", []))
n_load = len(grid_nodes.get("load", []))
n_shunt = len(grid_nodes.get("shunt", []))

print("Counts from JSON:", dict(n_bus=n_bus, n_gen=n_gen, n_load=n_load, n_shunt=n_shunt))
print("Counts from graph meta:", {k: meta[k] for k in ["n_bus","n_gen","n_load","n_shunt"]})

# 示例：取 JSON 的第0个 bus，看看在 x 的第0行的 raw 前几维是否一致（padding后）
if n_bus > 0:
    bus0 = np.array(grid_nodes["bus"][0], dtype=float)
    print("\nJSON bus[0] raw:", bus0)
    print("Graph x[0] raw :", raw_x[0, :len(bus0)])

# 示例：取 JSON 的第0个 generator，映射到 x 的 [n_bus + 0]
if n_gen > 0:
    gen0 = np.array(grid_nodes["generator"][0], dtype=float)
    print("\nJSON gen[0] raw:", gen0[:10], "...")
    print("Graph x[n_bus+0] raw:", raw_x[n_bus + 0, :len(gen0)])

Counts from JSON: {'n_bus': 14, 'n_gen': 5, 'n_load': 11, 'n_shunt': 1}
Counts from graph meta: {'n_bus': 14, 'n_gen': 5, 'n_load': 11, 'n_shunt': 1}

JSON bus[0] raw: [1.   3.   0.94 1.06]
Graph x[0] raw : [1.   3.   0.94 1.06]

JSON gen[0] raw: [1.000000e+02 1.700000e+00 0.000000e+00 3.400000e+00 5.000000e-02
 0.000000e+00 1.000000e-01 1.000000e+00 0.000000e+00 7.920951e+02] ...
Graph x[n_bus+0] raw: [1.000000e+02 1.700000e+00 0.000000e+00 3.400000e+00 5.000000e-02
 0.000000e+00 1.000000e-01 1.000000e+00 0.000000e+00 7.920951e+02
 0.000000e+00]


In [7]:
builder_norm = PowerGraphBuilder(normalize_features=True, include_solution=True, include_links=True)
g_norm = builder_norm.build_graph_from_json(SAMPLE_JSON)

if isinstance(g_norm, dict):
    x_norm = np.array(g_norm["x"], dtype=float)
    e_norm = np.array(g_norm["edge_attr"], dtype=float)
else:
    x_norm = g_norm.x.detach().cpu().numpy()
    e_norm = g_norm.edge_attr.detach().cpu().numpy()

print("Raw x mean/std  :", raw_x.mean(), raw_x.std())
print("Norm x mean/std :", x_norm[:, :NODE_RAW_MAX].mean(), x_norm[:, :NODE_RAW_MAX].std())

print("Raw e mean/std  :", raw_e.mean(), raw_e.std())
print("Norm e mean/std :", e_norm[:, :EDGE_RAW_MAX].mean(), e_norm[:, :EDGE_RAW_MAX].std())

Raw x mean/std  : 10.840602460709954 133.22232342352902
Norm x mean/std : 2.0837030374196486e-17 0.9045340337332908
Raw e mean/std  : 0.30153737100737105 0.8837559809230259
Norm e mean/std : 0.0 0.904534033733291
